In [1]:
import sys
from pathlib import Path

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

# Add the root directory to the `PYTHONPATH` to use the `recsys` Python module from the notebook.
if root_dir not in sys.path:
    sys.path.append(root_dir)
print(f"Added the following directory to the PYTHONPATH: {root_dir}")
    
# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

Local environment
Added the following directory to the PYTHONPATH: /Users/hongjiang/git/mlfs-book
HopsworksSettings initialized!


# Daily Feature Pipeline for Grass Pollen (Stockholm)

## Sections:
1. Fetch Pollen Data from Pollenrapporten API
2. Insert into Feature Group

**Schedule this notebook to run daily during pollen season (May-August)**

In [2]:
import datetime
import pandas as pd
import hopsworks
from mlfs.airquality import util
import warnings
import json
warnings.filterwarnings("ignore")

## Connect to Hopsworks

In [3]:
project = hopsworks.login(engine="python")
fs = project.get_feature_store()
secrets = hopsworks.get_secrets_api()

location_str = secrets.get_secret("SENSOR_LOCATION_JSON").value
location = json.loads(location_str)
country=location['country']
city=location['city']
street=location['street']

# Stockholm coordinates
latitude = location['latitude']
longitude = location['longitude']

today = datetime.date.today()
print(f"Fetching pollen data for: {today}")

2025-12-28 21:09:31,017 INFO: Initializing external client
2025-12-28 21:09:31,018 INFO: Base URL: https://c.app.hopsworks.ai:443


2025-12-28 21:09:32,669 INFO: Python Engine initialized.

Logged in to project, explore it here https://c.app.hopsworks.ai:443/p/1292436
Fetching pollen data for: 2025-12-28


## Get Feature Group Reference

In [4]:
grass_pollen_fg = fs.get_feature_group(
    name='grass_pollen',
    version=1
)

weather_fg = fs.get_feature_group(
    name='weather',
    version=1,
)

## Fetch Today's Pollen Data

In [5]:
# Fetch last 7 days to ensure we get today's data
start_date = (today - datetime.timedelta(days=7)).strftime('%Y-%m-%d')
end_date = today.strftime('%Y-%m-%d')

pollen_df = util.get_historical_pollen(
    start_date=start_date,
    end_date=end_date
)

if not pollen_df.empty:
    # Get only today's data
    pollen_df['date'] = pd.to_datetime(pollen_df['date']).dt.date
    pollen_today = pollen_df[pollen_df['date'] == today]
    pollen_today['date'] = pd.to_datetime(pollen_today['date'])
    print(f"Pollen data retrieved: {len(pollen_today)} records")
    print(pollen_today)
else:
    print("No pollen data available (likely outside monitoring season)")
    pollen_today = pd.DataFrame()

No pollen data available for 2025-12-21 to 2025-12-28
No pollen data available (likely outside monitoring season)


Get Weather Forecast data

In [6]:
hourly_df = util.get_hourly_weather_forecast(city, latitude, longitude)
hourly_df = hourly_df.set_index('date')

# We will only make 1 daily prediction, so we will replace the hourly forecasts with a single daily forecast
# We only want the daily weather data, so only get weather at 12:00
daily_df = hourly_df.between_time('11:59', '12:01')
daily_df = daily_df.reset_index()
daily_df['date'] = pd.to_datetime(daily_df['date']).dt.date
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df['city'] = city
daily_df

Coordinates 59.25°N 18.0°E
Elevation 24.0 m asl
Timezone None None
Timezone difference to GMT+0 0 s


,date,temperature_2m_mean,precipitation_sum,wind_speed_10m_max,wind_direction_10m_dominant,city
0,2025-12-28,1.90,0.0,16.279802,305.095886,Stockholm
1,2025-12-29,-0.85,0.0,22.473343,305.217682,Stockholm
2,2025-12-30,-1.95,0.0,25.387020,341.821899,Stockholm
3,2025-12-31,-4.35,0.0,5.692100,341.564941,Stockholm
4,2026-01-01,0.35,2.3,28.822491,152.474869,Stockholm
5,2026-01-02,0.50,0.3,13.276144,40.601215,Stockholm
6,2026-01-03,-5.60,0.0,12.594856,300.963684,Stockholm


## Upload to Feature Store

In [7]:
if not pollen_today.empty:
    grass_pollen_fg.insert(pollen_today, wait=True)
    print("✅ Pollen data uploaded successfully")
else:
    print("⚠️ No data to upload")

⚠️ No data to upload


In [8]:
# Insert new data
weather_fg.insert(daily_df, wait=True)

2025-12-28 21:10:27,627 INFO: 	2 expectation(s) included in expectation_suite.
Validation succeeded.
Validation Report saved successfully, explore a summary at https://c.app.hopsworks.ai:443/p/1292436/fs/1265790/fg/1737105


Uploading Dataframe: 100.00% |█| Rows 7/7 | Elapsed Time: 00:01 | Remaining Time


Launching job: weather_1_offline_fg_materialization
Job started successfully, you can follow the progress at 
https://c.app.hopsworks.ai:443/p/1292436/jobs/named/weather_1_offline_fg_materialization/executions
2025-12-28 21:10:46,016 INFO: Waiting for execution to finish. Current state: INITIALIZING. Final status: UNDEFINED
2025-12-28 21:10:55,947 INFO: Waiting for execution to finish. Current state: AGGREGATING_LOGS. Final status: FAILED
2025-12-28 21:10:56,138 INFO: Waiting for log aggregation to finish.
2025-12-28 21:12:56,845 ERROR: Execution failed with status: FAILED. See the logs for more information.


(Job('weather_1_offline_fg_materialization', 'SPARK'),
 {
   "success": true,
   "results": [
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_min_to_be_between",
         "kwargs": {
           "column": "precipitation_sum",
           "min_value": -0.1,
           "max_value": 1000.0,
           "strict_min": true
         },
         "meta": {
           "expectationId": 768053
         }
       },
       "result": {
         "observed_value": 0.0,
         "element_count": 7,
         "missing_count": null,
         "missing_percent": null
       },
       "meta": {
         "ingestionResult": "INGESTED",
         "validationTime": "2025-12-28T08:10:27.000626Z"
       },
       "exception_info": {
         "raised_exception": false,
         "exception_message": null,
         "exception_traceback": null
       }
     },
     {
       "success": true,
       "expectation_config": {
         "expectation_type": "expect_column_